**AI/BI Dashboard Definitions**

 **HOW TO USE:**

 In Databricks UI → Dashboards → New Dashboard → Import
 
 Paste each YAML block, OR create dashboards manually using
 the widget specs and SQL queries defined below.

 Each dashboard has:
   - Dataset SQL (what powers the widgets)
   - Widget specs (chart type, axes, filters)
   - Layout hints (row/column positions)

**DASHBOARD 1: Complaint Heatmap**

In [0]:
# Create: Dashboards → New → name "CivicOps: Complaint Heatmap"
# Add these widgets:

# Widget 1.1 — Heatmap: Complaints by Borough × Month
# Chart type: Heatmap
# Dataset SQL:
"""
SELECT
    borough,
    DATE_FORMAT(period, 'MMM yyyy')     AS month_label,
    SUM(total_complaints)               AS complaints
FROM civicops.gold.dash_heatmap
WHERE year >= YEAR(CURRENT_DATE()) - 1
GROUP BY borough, period, month_label
ORDER BY period, borough
"""
# X-axis: month_label
# Y-axis: borough
# Color: complaints (low=blue, high=red)

# Widget 1.2 — Bar Chart: Top 10 Complaint Types
# Chart type: Horizontal Bar
# Dataset SQL:
"""
SELECT
    complaint_type,
    SUM(total_complaints)               AS total,
    SUM(unresolved)                     AS unresolved,
    ROUND(SUM(unresolved) * 100.0 / SUM(total_complaints), 1) AS unresolved_pct
FROM civicops.gold.dash_heatmap
GROUP BY complaint_type
ORDER BY total DESC
LIMIT 10
"""
# X-axis: total | Color split: unresolved vs resolved | Y-axis: complaint_type

# Widget 1.3 — Scorecard: Total Open Complaints
# Chart type: Counter
# Dataset SQL:
"""
SELECT
    SUM(unresolved)                     AS open_complaints,
    SUM(total_complaints)               AS total_complaints,
    ROUND(SUM(unresolved) * 100.0 / SUM(total_complaints), 1) AS open_pct
FROM civicops.gold.dash_heatmap
"""

# Widget 1.4 — Line Chart: Daily Complaint Volume (last 30 days)
# Chart type: Line
# Dataset SQL:
"""
SELECT
    period,
    borough,
    SUM(total_complaints)               AS complaints
FROM civicops.gold.dash_heatmap
WHERE period >= ADD_MONTHS(CURRENT_DATE(), -6)
GROUP BY period, borough
ORDER BY period
"""
# X-axis: period | Y-axis: complaints | Series: borough

**DASHBOARD 2: Ward-wise Trends**

In [0]:
# Create: Dashboards → New → name "CivicOps: Ward-wise Trends"

# Widget 2.1 — Line Chart: Complaint Trend by Borough
# Chart type: Line
# Dataset SQL:
"""
SELECT
    borough,
    period,
    complaint_count,
    unresolved,
    avg_days_open,
    mom_pct_change
FROM civicops.gold.dash_ward_trends
ORDER BY period, borough
"""
# X-axis: period | Y-axis: complaint_count | Series: borough
# Add filter: borough (multi-select dropdown)

# Widget 2.2 — Bar Chart: MoM Change by Borough
# Chart type: Bar (positive/negative colors)
# Dataset SQL:
"""
SELECT
    borough,
    period,
    mom_change,
    mom_pct_change
FROM civicops.gold.dash_ward_trends
WHERE period >= ADD_MONTHS(CURRENT_DATE(), -3)
ORDER BY period, mom_change DESC
"""
# X-axis: borough | Y-axis: mom_pct_change | Color: green if <0, red if >0

# Widget 2.3 — Table: Borough Summary
# Chart type: Table
# Dataset SQL:
"""
SELECT
    borough,
    SUM(complaint_count)                AS total_complaints,
    SUM(unresolved)                     AS total_unresolved,
    ROUND(AVG(avg_days_open), 1)        AS avg_days_open,
    ROUND(AVG(mom_pct_change), 1)       AS avg_mom_growth_pct
FROM civicops.gold.dash_ward_trends
GROUP BY borough
ORDER BY total_unresolved DESC
"""

# Widget 2.4 — Pie Chart: Complaint Share by Borough (current month)
# Chart type: Pie
# Dataset SQL:
"""
SELECT
    borough,
    SUM(complaint_count)                AS complaints
FROM civicops.gold.dash_ward_trends
WHERE year = YEAR(CURRENT_DATE())
  AND month = MONTH(CURRENT_DATE())
GROUP BY borough
"""


**DASHBOARD 3: SLA Analytics**

In [0]:
# Create: Dashboards → New → name "CivicOps: SLA Analytics"

# Widget 3.1 — Scorecard Row: Key SLA Metrics
# Chart type: Counter (create 3 separate counter widgets)
# Dataset SQL for each:

# SLA Health Score (avg across all depts):
"""
SELECT ROUND(AVG(sla_health_score), 1) AS overall_sla_health
FROM civicops.gold.dash_sla_analytics
"""

# Total Breached:
"""
SELECT SUM(breached_count) AS total_breached
FROM civicops.gold.dash_sla_analytics
"""

# Breach Rate:
"""
SELECT ROUND(SUM(breached_count) * 100.0 / SUM(total_tickets), 1) AS overall_breach_rate_pct
FROM civicops.gold.dash_sla_analytics
"""

# Widget 3.2 — Bar Chart: Breach Rate by Department
# Chart type: Horizontal Bar
# Dataset SQL:
"""
SELECT
    dept_code,
    dept_name,
    SUM(total_tickets)                  AS tickets,
    SUM(breached_count)                 AS breached,
    ROUND(SUM(breached_count) * 100.0 / SUM(total_tickets), 1) AS breach_rate_pct,
    ROUND(AVG(sla_health_score), 1)     AS health_score
FROM civicops.gold.dash_sla_analytics
GROUP BY dept_code, dept_name
ORDER BY breach_rate_pct DESC
"""
# X-axis: breach_rate_pct | Y-axis: dept_name | Color: red gradient

# Widget 3.3 — Line Chart: SLA Breach Trend over Time
# Chart type: Line
# Dataset SQL:
"""
SELECT
    month,
    dept_code,
    SUM(breached_count)                 AS breaches,
    ROUND(SUM(breached_count) * 100.0 / SUM(total_tickets), 1) AS breach_rate_pct
FROM civicops.gold.dash_sla_analytics
GROUP BY month, dept_code
ORDER BY month
"""
# X-axis: month | Y-axis: breach_rate_pct | Series: dept_code

# Widget 3.4 — Heatmap: SLA Health by Dept × Severity
# Chart type: Heatmap
# Dataset SQL:
"""
SELECT
    dept_name,
    CONCAT('Severity ', severity_score) AS severity_label,
    ROUND(AVG(sla_health_score), 1)     AS health_score
FROM civicops.gold.dash_sla_analytics
GROUP BY dept_name, severity_score
ORDER BY severity_score DESC
"""
# X-axis: severity_label | Y-axis: dept_name | Color: health_score

**DASHBOARD 4: Department Performance**

In [0]:
# Create: Dashboards → New → name "CivicOps: Department Performance"

# Widget 4.1 — Table: Department Scorecard
# Chart type: Table with conditional formatting
# Dataset SQL:
"""
SELECT
    dept_code,
    dept_name,
    SUM(total_tickets)                  AS total_tickets,
    SUM(resolved_count)                 AS resolved,
    ROUND(AVG(resolution_rate_pct), 1)  AS resolution_rate_pct,
    SUM(escalated)                      AS escalated,
    ROUND(SUM(escalated) * 100.0 / SUM(total_tickets), 1) AS escalation_rate_pct,
    SUM(health_risk_tickets)            AS health_risk_tickets,
    ROUND(AVG(avg_severity), 2)         AS avg_severity,
    SUM(critical_high_count)            AS critical_high
FROM civicops.gold.dash_dept_performance
GROUP BY dept_code, dept_name
ORDER BY total_tickets DESC
"""
# Conditional formatting: resolution_rate_pct → green>80, yellow 50-80, red<50

# Widget 4.2 — Bar Chart: Tickets by Department
# Chart type: Stacked Bar
# Dataset SQL:
"""
SELECT
    dept_name,
    category,
    SUM(total_tickets)                  AS tickets
FROM civicops.gold.dash_dept_performance
GROUP BY dept_name, category
ORDER BY tickets DESC
"""
# X-axis: dept_name | Y-axis: tickets | Stack: category

# Widget 4.3 — Scatter: Resolution Rate vs Avg Severity
# Chart type: Scatter
# Dataset SQL:
"""
SELECT
    dept_name,
    ROUND(AVG(avg_severity), 2)         AS avg_severity,
    ROUND(AVG(resolution_rate_pct), 1)  AS resolution_rate_pct,
    SUM(total_tickets)                  AS total_tickets
FROM civicops.gold.dash_dept_performance
GROUP BY dept_name
"""
# X-axis: avg_severity | Y-axis: resolution_rate_pct
# Bubble size: total_tickets | Label: dept_name

# Widget 4.4 — Bar: Field Visit Requirement by Dept
# Chart type: Bar
# Dataset SQL:
"""
SELECT
    dept_name,
    SUM(field_visits_needed)            AS visits_needed,
    SUM(total_tickets)                  AS total,
    ROUND(SUM(field_visits_needed) * 100.0 / SUM(total_tickets), 1) AS visit_rate_pct
FROM civicops.gold.dash_dept_performance
GROUP BY dept_name
ORDER BY visit_rate_pct DESC
"""

**DASHBOARD 5: Escalation Insights**

In [0]:
# Create: Dashboards → New → name "CivicOps: Escalation Insights"

# Widget 5.1 — Scorecard Row
"""
-- Total escalations
SELECT COUNT(*) AS total_escalations FROM civicops.memory.escalation_context
"""
"""
-- Health risk escalations
SELECT SUM(health_risk_escalations) AS health_risk_escalations
FROM civicops.gold.dash_escalation_insights
"""
"""
-- Resolved after escalation rate
SELECT ROUND(
    SUM(resolved_after_escalation) * 100.0 / SUM(escalation_count), 1
) AS resolved_after_escalation_pct
FROM civicops.gold.dash_escalation_insights
"""

# Widget 5.2 — Sankey / Bar: Escalation Flow (from dept → to dept)
# Chart type: Bar (Sankey not natively available — use grouped bar)
# Dataset SQL:
"""
SELECT
    escalated_from_dept,
    escalated_to_dept,
    SUM(escalation_count)               AS escalations
FROM civicops.gold.dash_escalation_insights
GROUP BY escalated_from_dept, escalated_to_dept
ORDER BY escalations DESC
"""
# X-axis: escalated_from_dept | Y-axis: escalations | Color: escalated_to_dept

# Widget 5.3 — Line Chart: Escalation Trend over Time
# Chart type: Line
# Dataset SQL:
"""
SELECT
    month,
    SUM(escalation_count)               AS escalations,
    SUM(health_risk_escalations)        AS health_risk,
    SUM(infra_risk_escalations)         AS infra_risk,
    SUM(resolved_after_escalation)      AS resolved_after
FROM civicops.gold.dash_escalation_insights
GROUP BY month
ORDER BY month
"""
# X-axis: month | Y-axis: escalations | Multi-line: health_risk, infra_risk

# Widget 5.4 — Bar: Escalations by Category
# Chart type: Horizontal Bar
# Dataset SQL:
"""
SELECT
    category,
    SUM(escalation_count)               AS total_escalations,
    ROUND(AVG(avg_resolution_hrs), 1)   AS avg_resolution_hrs,
    SUM(health_risk_escalations)        AS health_risk_count
FROM civicops.gold.dash_escalation_insights
GROUP BY category
ORDER BY total_escalations DESC
"""

# Widget 5.5 — Table: Recent Escalations Detail
# Chart type: Table
# Dataset SQL:
"""
SELECT
    e.ticket_id,
    e.escalated_at,
    e.escalated_from_dept,
    e.escalated_to_dept,
    e.escalation_reason,
    e.severity_at_escalation,
    e.health_risk,
    e.resolved_after_escalation,
    h.raw_complaint,
    h.category
FROM civicops.memory.escalation_context e
JOIN civicops.memory.complaint_history h ON e.ticket_id = h.ticket_id
ORDER BY e.escalated_at DESC
LIMIT 50
"""
